# Parte 4 — Modelo de Ensamble Avanzado

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    GroupKFold
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    log_loss,
    brier_score_loss
)

from xgboost import XGBClassifier

# ============================================================
# Configuración entorno
# ============================================================

venv_site_packages = os.path.abspath(
    os.path.join(os.getcwd(), '.venv', 'Lib', 'site-packages')
)

if (
    os.path.isdir(venv_site_packages)
    and venv_site_packages not in sys.path
):
    sys.path.insert(0, venv_site_packages)

    import site
    site.addsitedir(venv_site_packages)

RANDOM_STATE = 42

# ============================================================
# Cargar dataset
# ============================================================

try:

    data = pd.read_pickle(
        'dataframes/data_con_jugadores.pkl'
    )

except Exception:

    with open(
        'dataframes/data_con_jugadores.pkl',
        'rb'
    ) as f:

        data = pickle.load(f)

# ============================================================
# Conversión tipos
# ============================================================

for col in data.select_dtypes(
    include=['string', 'category']
).columns:

    data[col] = data[col].astype('object')

# ============================================================
# Convertir duración
# ============================================================

if 'duracion_punto' in data.columns:

    data['duracion_punto'] = pd.to_timedelta(
        data['duracion_punto'],
        errors='coerce'
    ).dt.total_seconds()

# ============================================================
# Variable objetivo
# ============================================================

TARGET_COL = 'ganador_partido'

# ============================================================
# Variables con leakage
# ============================================================

cols_excluir = [

    # Variable objetivo
    "ganador_partido",

    # Identificadores
    "partido",
    "punto",

    # Contexto administrativo
    "cancha",

    # Variables categóricas / texto
    "jugador_1_equipo",
    "jugador_2_equipo",
    "jugador_1_rival",
    "jugador_2_rival",

    # Variables categóricas adicionales
    "genero_jugador_1_equipo",
    "genero_jugador_2_equipo",
    "genero_jugador_1_rival",
    "genero_jugador_2_rival",

    # Duración sigue siendo object en muchos casos
    "duracion_punto"
]

# ============================================================
# Features y target
# ============================================================

X = data.drop(columns=cols_excluir)

y = data[TARGET_COL].astype(int)

# ============================================================
# Variables numéricas y categóricas
# ============================================================

columnas_numericas = X.select_dtypes(
    include=[
        'int64',
        'float64',
        'int32',
        'float32'
    ]
).columns.tolist()

columnas_categoricas = X.select_dtypes(
    include=[
        'object',
        'category',
        'bool',
        'string'
    ]
).columns.tolist()

print(f'\nVariables numéricas: {len(columnas_numericas)}')
print(f'Variables categóricas: {len(columnas_categoricas)}')

# ============================================================
# Preprocesamiento
# ============================================================

transformador_numerico = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    ),
    (
        'scaler',
        StandardScaler()
    )
])

transformador_categorico = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='most_frequent')
    ),
    (
        'encoder',
        OneHotEncoder(handle_unknown='ignore')
    )
])

preprocesador = ColumnTransformer([
    (
        'numerico',
        transformador_numerico,
        columnas_numericas
    ),
    (
        'categorico',
        transformador_categorico,
        columnas_categoricas
    )
])

# ============================================================
# Split train/test
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# ============================================================
# Modelo base
# ============================================================

modelo_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist'
)

pipeline = Pipeline([
    (
        'preprocesamiento',
        preprocesador
    ),
    (
        'modelo',
        modelo_base
    )
])

# ============================================================
# Hiperparámetros
# ============================================================

parametros = {

    'modelo__n_estimators': [
        200,
        300,
        400,
        500
    ],

    'modelo__max_depth': [
        3,
        4,
        5,
        6
    ],

    'modelo__learning_rate': [
        0.01,
        0.03,
        0.05,
        0.1
    ],

    'modelo__subsample': [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    'modelo__colsample_bytree': [
        0.6,
        0.7,
        0.8,
        0.9
    ],

    'modelo__gamma': [
        0,
        0.1,
        0.3
    ],

    'modelo__min_child_weight': [
        1,
        3,
        5
    ],

    'modelo__reg_alpha': [
        0,
        0.01,
        0.1
    ],

    'modelo__reg_lambda': [
        1,
        1.5,
        2
    ]
}

# ============================================================
# Validación cruzada
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ============================================================
# Búsqueda hiperparámetros
# ============================================================

busqueda = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parametros,
    n_iter=25,
    scoring='roc_auc',
    cv=cv,
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

# ============================================================
# Entrenamiento
# ============================================================

busqueda.fit(X_train, y_train)

mejor_modelo = busqueda.best_estimator_

# ============================================================
# Predicciones
# ============================================================

predicciones = mejor_modelo.predict(X_test)

probabilidades = mejor_modelo.predict_proba(X_test)

probabilidades_binarias = probabilidades[:, 1]

# ============================================================
# Métricas
# ============================================================

accuracy = accuracy_score(
    y_test,
    predicciones
)

precision = precision_score(
    y_test,
    predicciones,
    average='weighted'
)

recall = recall_score(
    y_test,
    predicciones,
    average='weighted'
)

f1 = f1_score(
    y_test,
    predicciones,
    average='weighted'
)

roc_auc = roc_auc_score(
    y_test,
    probabilidades_binarias
)

logloss = log_loss(
    y_test,
    probabilidades_binarias
)

brier = brier_score_loss(
    y_test,
    probabilidades_binarias
)

# ============================================================
# Resultados
# ============================================================

print('\n==============================')
print('MEJORES HIPERPARÁMETROS')
print('==============================')

print(busqueda.best_params_)

print('\n==============================')
print('MÉTRICAS')
print('==============================')

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC AUC   : {roc_auc:.4f}')
print(f'Log Loss  : {logloss:.4f}')
print(f'Brier     : {brier:.4f}')

print('\n==============================')
print('REPORTE CLASIFICACIÓN')
print('==============================')

print(
    classification_report(
        y_test,
        predicciones
    )
)

# ============================================================
# Guardar modelo
# ============================================================

joblib.dump(
    mejor_modelo,
    'xgboost_final.pkl'
)

# ============================================================
# Importancia variables
# ============================================================

modelo_xgb = mejor_modelo.named_steps['modelo']

importancias = modelo_xgb.feature_importances_

nombres = mejor_modelo.named_steps[
    'preprocesamiento'
].get_feature_names_out()

ranking = pd.DataFrame({
    'Variable': nombres,
    'Importancia': importancias
})

ranking = ranking.sort_values(
    by='Importancia',
    ascending=False
)

top20 = ranking.head(20)

# ============================================================
# PDF Reporte
# ============================================================

with PdfPages('reporte_xgboost.pdf') as pdf:

    # --------------------------------------------------------
    # Resumen
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.axis('off')

    resumen = (
        'MODELO AVANZADO - XGBOOST\n\n'
        f'Accuracy:  {accuracy:.4f}\n'
        f'Precision: {precision:.4f}\n'
        f'Recall:    {recall:.4f}\n'
        f'F1 Score:  {f1:.4f}\n'
        f'ROC AUC:   {roc_auc:.4f}\n'
        f'Log Loss:  {logloss:.4f}\n'
        f'Brier:     {brier:.4f}\n\n'
        f'Mejores parámetros:\n'
        f'{busqueda.best_params_}'
    )

    ax.text(
        0.01,
        0.95,
        resumen,
        fontsize=12,
        va='top'
    )

    pdf.savefig(fig, bbox_inches='tight')

    plt.close()

    # --------------------------------------------------------
    # Matriz confusión
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 6))

    matriz = confusion_matrix(
        y_test,
        predicciones
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz
    )

    disp.plot(ax=ax)

    ax.set_title('Matriz de Confusion')

    pdf.savefig(fig, bbox_inches='tight')

    plt.close()

    # --------------------------------------------------------
    # ROC
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 6))

    RocCurveDisplay.from_predictions(
        y_test,
        probabilidades_binarias,
        ax=ax
    )

    ax.set_title('Curva ROC')

    pdf.savefig(fig, bbox_inches='tight')

    plt.close()

    # --------------------------------------------------------
    # Precision Recall
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 6))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        probabilidades_binarias,
        ax=ax
    )

    ax.set_title('Curva Precision Recall')

    pdf.savefig(fig, bbox_inches='tight')

    plt.close()

    # --------------------------------------------------------
    # Importancia variables
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10, 8))

    ax.barh(
        top20['Variable'][::-1],
        top20['Importancia'][::-1]
    )

    ax.set_title(
        'Top 20 Variables Mas Importantes'
    )

    ax.set_xlabel('Importancia')

    pdf.savefig(fig, bbox_inches='tight')

    plt.close()

print('\nArchivos generados correctamente:')
print('xgboost_final.pkl')
print('reporte_xgboost.pdf')


Variables numéricas: 84
Variables categóricas: 0
Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.3, modelo__learning_rate=0.03, modelo__max_depth=3, modelo__min_child_weight=3, modelo__n_estimators=400, modelo__reg_alpha=0.1, modelo__reg_lambda=1, modelo__subsample=1.0; total time=   1.2s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.3, modelo__learning_rate=0.03, modelo__max_depth=3, modelo__min_child_weight=3, modelo__n_estimators=400, modelo__reg_alpha=0.1, modelo__reg_lambda=1, modelo__subsample=1.0; total time=   1.3s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.3, modelo__learning_rate=0.03, modelo__max_depth=3, modelo__min_child_weight=3, modelo__n_estimators=400, modelo__reg_alpha=0.1, modelo__reg_lambda=1, modelo__subsample=1.0; total time=   1.3s
[CV] END modelo__colsample_bytree=0.6, modelo__gamma=0.3, modelo__learning_rate=0.03, modelo__max_depth=3, modelo__min_child_weight=3, modelo__n